# Decision Trees.

---

## Importing Libraries:

In [1]:
import time

import numpy as np

import pandas as pd

from sklearn.model_selection import train_test_split, GridSearchCV, cross_val_score
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer, TransformedTargetRegressor
from sklearn.pipeline import Pipeline
from sklearn.tree import DecisionTreeRegressor
from sklearn.metrics import r2_score, mean_absolute_percentage_error

## Importing Dataset:

In [2]:
data = pd.read_csv("Cleaned Laptop Dataset.csv")

In [3]:
data.drop(columns=['Unnamed: 0'], inplace=True)

## Building Model:

In [4]:
X = data.drop(columns=['Price'])
y = data['Price']

X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=69)

In [5]:
preprocessing = ColumnTransformer(transformers=[
    ('encode', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series'])
], remainder='passthrough')

X_train = preprocessing.fit_transform(X_train)
X_test = preprocessing.transform(X_test)

In [6]:
param = {'criterion' : ['squared_error', 'absolute_error'],
        'max_depth' : [1, 2, 3, 4, 5, 6, 7, 8, None]}

grid = GridSearchCV(
    estimator=DecisionTreeRegressor(),
    param_grid=param,
    scoring='r2',
    n_jobs=-1,
    cv=5
)

grid.fit(X_train, y_train)

print(f"Best Parameters: {grid.best_params_}")
print(f"Best Score: {grid.best_score_ * 100:.2f}%")

Best Parameters: {'criterion': 'absolute_error', 'max_depth': 7}
Best Score: 75.90%


In [7]:
dct = DecisionTreeRegressor(criterion='absolute_error', max_depth=7)

start_time = time.time()
dct.fit(X_train, y_train)
end_time = time.time() - start_time

print('Model Trained Sucessfully!')
print(f'Time Taken For Training: {end_time:.2f}s')

Model Trained Sucessfully!
Time Taken For Training: 0.62s


## Results:

In [8]:
def adjusted_r2_score(y, y_pred):

    r2 = r2_score(y, y_pred)
    n = X.shape[0]
    k = X.shape[1]

    return 1 - ((1 - r2) * (n - 1) / (n - 1 - k))

In [9]:
# Results on Training Data:

y_pred_train = dct.predict(X_train)

print(f"r2 Score (Trainig Data): {r2_score(y_train, y_pred_train) * 100 :.2f}%")
print(f"\nmean absolute error(Training Data): {mean_absolute_percentage_error(y_train, y_pred_train) * 100 :.2f}%")
print(f"\nadjusted r2 Score: {adjusted_r2_score(y_train, y_pred_train) * 100 :.2f}%")

r2 Score (Trainig Data): 80.65%

mean absolute error(Training Data): 17.91%

adjusted r2 Score: 80.59%


In [10]:
# Results on Testing Data:

y_pred_test = dct.predict(X_test)

print(f"r2 Score (Testing Data): {r2_score(y_test, y_pred_test) * 100 :.2f}%")
print(f"\nmean absolute error (Testing Data): {mean_absolute_percentage_error(y_test, y_pred_test) * 100 :.2f}%")
print(f"\nadjusted r2 Score: {adjusted_r2_score(y_test, y_pred_test) * 100 :.2f}%")

r2 Score (Testing Data): 77.98%

mean absolute error (Testing Data): 19.89%

adjusted r2 Score: 77.91%


## Cross-Val-Score:

In [11]:
preprocessing = ColumnTransformer(transformers=[
    ('encode', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series'])
], remainder='passthrough')

dct_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', DecisionTreeRegressor(criterion='absolute_error', max_depth=7))
])

In [12]:
print(f"r2 Score: {np.mean(cross_val_score(estimator=dct_pipeline, X=X, y=y, scoring='r2', cv=10, n_jobs=-1)) * 100 :.2f}%")

r2 Score: 76.03%


In [13]:
print(f"mean absolute error: {-np.mean(cross_val_score(estimator=dct_pipeline, X=X, y=y, scoring='neg_mean_absolute_percentage_error', cv=10, n_jobs=-1)) * 100 :.2f}%")

mean absolute error: 20.17%


---

# Experimentation:

- ## Transformation (Target Feature):

In [14]:
preprocessing = ColumnTransformer(transformers=[
    ('encode', OneHotEncoder(drop='first', handle_unknown='ignore'), ['Display Type', 'Display Touchscreen', 'RAM Type', 'GPU Brand', 'GPU Segment', 'Brand', 'Operating System', 'CPU Brand', 'CPU Segment', 'CPU Series'])
], remainder='passthrough')

dct_pipeline = Pipeline(steps=[
    ('preprocessing', preprocessing),
    ('model', DecisionTreeRegressor())
])

dct_model = TransformedTargetRegressor(
    regressor=dct_pipeline,
    func=np.log1p,
    inverse_func=np.expm1
)

param = {'regressor__model__criterion' : ['squared_error', 'absolute_error'],
        'regressor__model__max_depth' : [1, 2, 3, 4, 5, 6, 7, 8, None]}

grid_model = GridSearchCV(
    estimator=dct_model,
    param_grid=param,
    scoring='r2',
    n_jobs=-1,
    cv=5
)

print(f"r2 Score: {np.mean(cross_val_score(estimator=grid_model, X=X, y=y, scoring='r2', cv=10, n_jobs=-1)) * 100 :.2f}%")

print(f"\nmean absolute error: {-np.mean(cross_val_score(estimator=grid_model, X=X, y=y, scoring='neg_mean_absolute_percentage_error', cv=10, n_jobs=-1)) * 100 :.2f}%")

r2 Score: 76.03%

mean absolute error: 19.82%


---

# Conclusion:

*Decision Trees* performs lowers compared to *SVM*, *KNN*, *Linear Regression* with an **r2-Score = 76.03% & M.P.A.E = 19.82%**.